In [ ]:
import ee
from Sentinel_Rain_Detection.data_downloads import build_export_image
import json
import geemap
import ipyleaflet
import ipywidgets as widgets
from datetime import date
ee.Authenticate()
ee.Initialize()

## Settings


In [ ]:
geojson_path = 'data/geojson/TA00251_fields.geojson'
area_id = 'TA00251'
field_name = 'Field16'

start_date = '2026-08-01'
end_date = '2026-09-01'

orbit = 'ASCENDING'

# Sentinel-1 download

### Load fields from geojson

In [ ]:
with open(geojson_path) as f:
    data_fields = json.load(f)

if data_fields['type'] == 'FeatureCollection':
    features_fields = [
        ee.Feature(f['geometry'], f.get('properties', {}))
        for f in data_fields['features']
    ]
    fields = ee.FeatureCollection(features_fields)
else:
    geom_raw = ee.Geometry(data_fields['geometry'])
    fields = ee.FeatureCollection([ee.Feature(geom_raw, data_fields.get('properties', {}))])

field_selected = fields.filter(ee.Filter.eq('id', field_name)).first()
geom = field_selected.geometry()

region_coords = geom.bounds().getInfo()['coordinates']

### Filter Sentinel-1 images with Google Earth Engine

In [ ]:
s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterDate(start_date, end_date)
    .filterBounds(geom)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', orbit))
    .filter(ee.Filter.eq('resolution_meters', 10))
    .select(['VV', 'VH', 'angle'])
)

n_images = s1.size().getInfo()
print(f"Images in collection: {n_images}")

s1_list = s1.toList(n_images)

### Download to Google Drive folder

In [ ]:
drive_folder = f"S1_{area_id}_{field_name}"


for i in range(n_images):

    img_raw = ee.Image(s1_list.get(i)).clip(geom)
    img = build_export_image(img_raw)

    date_str = img.date().format('YYYYMMdd').getInfo()
    name = f"S1_{field_name}_{date_str}"

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=name,
        folder=drive_folder,
        fileNamePrefix=name,
        region=region_coords,
        scale=10,
        maxPixels=1e13,
    )
    task.start()
    print(f"Started export: {name}")

# Sentinel-1 and 2 in map

In [ ]:
ee.Initialize()

# Map setup
Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map.centerObject(fields)
Map.addLayer(fields, {'color': 'blue'}, f"{area_id} Fields")

# Widgets
start_date = widgets.DatePicker(description='Start', value=date(2017, 1, 1))
end_date = widgets.DatePicker(description='End', value=date(2025, 9, 30))

satellite_selector = widgets.Dropdown(
    options=['Sentinel-2', 'Sentinel-1'], value='Sentinel-2', description='Satellite:'
)
pol_selector = widgets.Dropdown(
    options=['VV', 'VH', 'VV/VH RGB'], value='VV', description='Polarization:'
)
load_button = widgets.Button(description='Load Image', button_style='success', icon='refresh')
output = widgets.Output()

viridis_palette = [
    '#440154', '#482777', '#3E4989', '#31688E', '#26828E',
    '#1F9E89', '#35B779', '#6CCE59', '#B4DE2C', '#FDE725',
]

S1_VIS = {'min': 0, 'max': 0.2, 'palette': viridis_palette}


def to_linear(img_db, band_name):
    return img_db.expression("10 ** (x / 10)", {"x": img_db}).rename(band_name)


def load_sentinel2(s_date, e_date):
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(s_date, e_date)
        .filterBounds(fields)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
    )
    image = collection.median().clip(fields)
    vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
    layer_name = f"Sentinel-2 {s_date} → {e_date}"

    Map.addLayer(image, vis_params, layer_name)
    print(f"✅ Sentinel-2 loaded from {s_date} to {e_date}")


def load_sentinel1(s_date, e_date, pol):
    collection = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterDate(s_date, e_date)
        .filterBounds(fields)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
    )

    count = collection.size().getInfo()
    if count == 0:
        print("⚠️ No Sentinel-1 data for this range.")
        return

    if pol == 'VV/VH RGB':
        vv_lin = to_linear(collection.select('VV').median(), 'VV')
        vh_lin = to_linear(collection.select('VH').median(), 'VH')
        ratio = vv_lin.divide(vh_lin).rename('VVVH_ratio')
        image = vv_lin.addBands(vh_lin).addBands(ratio).clip(fields)
        vis_params = {**S1_VIS, 'bands': ['VV', 'VH', 'VVVH_ratio']}
        layer_name = f"Sentinel-1 VV/VH RGB (linear) {s_date} → {e_date}"
    else:
        pol_collection = collection.filter(
            ee.Filter.listContains('transmitterReceiverPolarisation', pol)
        )
        image = to_linear(pol_collection.select(pol).median(), pol).clip(fields)
        vis_params = {**S1_VIS, 'bands': [pol]}
        layer_name = f"Sentinel-1 {pol} (linear) {s_date} → {e_date}"

    Map.addLayer(image, vis_params, layer_name)
    Map.add_colorbar(vis_params=vis_params, label=f"σ⁰ {pol} (linear)", layer_name=layer_name)
    print(f"✅ Sentinel-1 ({pol}) in *linear* scale loaded ({count} scenes)")


def load_image(b):
    with output:
        output.clear_output()
        print("🔄 Loading images...")

        s_date = start_date.value.strftime('%Y-%m-%d')
        e_date = end_date.value.strftime('%Y-%m-%d')

        if satellite_selector.value == 'Sentinel-2':
            load_sentinel2(s_date, e_date)
        else:
            load_sentinel1(s_date, e_date, pol_selector.value)


load_button.on_click(load_image)

controls = widgets.VBox([satellite_selector, pol_selector, start_date, end_date, load_button, output])
widgets.VBox([controls, Map])